# Paper fixes: boolean paired intervals + second untrained floor init

**Runtime → GPU. Run all.** Part 1 (boolean) is CPU over cached stores;
Part 2 (second floor init) is the only GPU work. Roughly 45 minutes total.

Two review findings, one run each:

**Part 1** resolves the InterpScience paper's weakest row. The boolean
probe-vs-masked-line differences ($-0.001$ to $+0.005$) have no paired
uncertainty because aligned predictions were not retained. `probe.py` and
`baselines.py` now both emit per-occurrence predictions, so re-running the
three Python probes and the surface baseline on the same frozen sample gives
`bootstrap_ci.py delta` everything it needs.

**Part 2** addresses the LP4FM fragility: the untrained context-pooled floor
rests on one weight initialization. A second seed either confirms the floor or
widens what the paper must claim.

In [ ]:
# 1 - setup
import pathlib, os, subprocess
BRANCH = "main"
REPO = "/content/code-model-interpretability"
if not pathlib.Path(REPO).exists():
    !git clone -q https://github.com/nolanlwin/code-model-interpretability.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q -B {BRANCH} origin/{BRANCH} && git pull -q
!git log --oneline -1
!pip install -q transformers==5.8.0 torch numpy scikit-learn tree_sitter \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
try:
    drive.mount("/content/drive")
except Exception as e:
    print(f"retrying mount ({e})")
    drive.mount("/content/drive", force_remount=True)

XL = "/content/drive/MyDrive/code-model-interpretability/xlcost"      # boolean-era artifacts
MK = "/content/drive/MyDrive/code-model-interpretability/masked"       # masked-probe artifacts
!mkdir -p data/xlcost outputs/xlcost_occ outputs/activations_xlcost outputs/probe_results outputs/crosslang outputs/role_occ
!cp -rn {XL}/stores/* outputs/activations_xlcost/ 2>/dev/null || true
!cp -n  {XL}/xlcost_occ/* outputs/xlcost_occ/ 2>/dev/null || true
!cp -n  {XL}/data_xlcost/* data/xlcost/ 2>/dev/null || true
!cp -rn {MK}/stores/* outputs/activations_xlcost/ 2>/dev/null || true
!cp -n  {MK}/role_occ/* outputs/role_occ/ 2>/dev/null || true
!cp -n  {MK}/data_xlcost/* data/xlcost/ 2>/dev/null || true

# Capability check: the flags this notebook passes must exist on this ref.
helps = ""
for argv in (["scripts/baselines.py","run","--help"],
             ["scripts/probe.py","run","--help"],
             ["scripts/extract_activations.py","run","--help"]):
    r = subprocess.run(["python"]+argv, capture_output=True, text=True)
    helps += r.stdout + r.stderr
need = ["--primary-feature", "--sample-ids", "--random-seed", "--pool"]
missing = [f for f in need if f not in helps]
if missing:
    raise SystemExit(f"BRANCH={BRANCH!r} lacks {missing}; use the branch that has them.")
import torch
print(f"setup ok | cuda={torch.cuda.is_available()}")

In [ ]:
# 2 - Part 1 guard: the stores must be the CORRECTED extraction (6,361
#     Python records), not the pre-fix one (16,671). A stale store would
#     reproduce nothing and quietly invalidate the pairing.
import json
MODELS = {"qwen2515b": "Qwen/Qwen2.5-1.5B",
          "qwen25coder15b": "Qwen/Qwen2.5-Coder-1.5B",
          "starcoder27b": "bigcode/starcoder2-7b"}
EXPECT = {}
for slug in MODELS:
    art = json.load(open(f"results/boolean/probe/python_train_{slug}_problem.json"))
    EXPECT[slug] = {"records": art["load_stats"]["records"],
                    "f1": art["aggregate"]["test_macro_f1_mean"]}
    store = f"outputs/activations_xlcost/python_train_{slug}"
    if not pathlib.Path(f"{store}/meta.json").exists():
        raise SystemExit(f"missing store {store}; check {{XL}}/stores on Drive")
    n = sum(1 for _ in open(f"{store}/index.jsonl"))
    want = art["load_stats"]["records"] + art["load_stats"].get("index_skipped_rows", 0)
    if n != want:
        raise SystemExit(f"{store}: index has {n} rows, artifact expects {want} -- "
                         "this is the WRONG (stale) store; do not proceed")
    print(f"  {slug}: store matches committed artifact ({n} rows), "
          f"target F1 {EXPECT[slug]['f1']:.4f}")

In [ ]:
# 3 - Part 1: re-probe with predictions retained, and check we reproduced
#     the committed number before trusting anything downstream.
for slug in MODELS:
    out = f"outputs/probe_results/python_train_{slug}_paired.json"
    if pathlib.Path(out).exists():
        continue
    !python scripts/probe.py run --store outputs/activations_xlcost/python_train_{slug} \
      --split-policy repo --allow-class-drop --control-task --output {out}
for slug in MODELS:
    d = json.load(open(f"outputs/probe_results/python_train_{slug}_paired.json"))
    got, want = d["aggregate"]["test_macro_f1_mean"], EXPECT[slug]["f1"]
    print(f"  {slug}: rerun F1 {got:.4f} vs committed {want:.4f} "
          f"({'REPRODUCED' if abs(got-want) < 2e-3 else 'DIFFERS -- investigate before using'})")
    n = len(d.get("test_predictions") or [])
    if n == 0:
        raise SystemExit(f"{slug}: no test_predictions emitted; wrong probe.py version")
    print(f"          {n} per-occurrence predictions retained")

In [ ]:
# 4 - Part 1: masked-line surface baseline on the SAME occurrences, with
#     predictions, pinned to line_masked (the family the paper's 0.983 is).
for slug in MODELS:
    out = f"outputs/probe_results/python_train_{slug}_surface_paired.json"
    if pathlib.Path(out).exists():
        continue
    !python scripts/baselines.py run \
      --occurrences outputs/xlcost_occ/python_train.jsonl \
      --canonical data/xlcost/python_train.jsonl \
      --sample-ids outputs/probe_results/python_train_{slug}_paired.json.sample_ids.json \
      --primary-feature line_masked \
      --output {out}
print("surface done")

In [ ]:
# 5 - Part 1: the paired intervals the paper could not state.
import csv
rows = []
for slug in MODELS:
    r = subprocess.run(["python","scripts/bootstrap_ci.py","delta",
        f"outputs/probe_results/python_train_{slug}_paired.json",
        f"outputs/probe_results/python_train_{slug}_surface_paired.json",
        "--n-boot","2000"], capture_output=True, text=True)
    print(f"=== {slug} (probe minus masked line) ===")
    print(r.stdout.strip() or r.stderr.strip())
    out = json.loads(r.stdout) if r.stdout.strip().startswith("{") else None
    if out:
        rows.append({"model": MODELS[slug], "slug": slug, **{k: out[k] for k in
                     ("delta","ci_low","ci_high","excludes_zero","n_clusters") if k in out}})
if rows:
    with open("results/boolean/probe/paired_probe_vs_line.csv","w",newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    print("wrote results/boolean/probe/paired_probe_vs_line.csv")
!mkdir -p {XL}/probe_results && cp outputs/probe_results/*_paired*.json {XL}/probe_results/ 2>/dev/null || true

In [ ]:
# 6 - Part 2: second untrained context-pooled floor (seed 1). GPU, ~25 min.
import re, itertools
MODEL = "Qwen/Qwen2.5-Coder-1.5B"
LANGS = {"Python": "python", "Javascript": "javascript", "PHP": "php"}
SPLIT = "train"; CT = 16
slug = lambda mid: re.sub(r"[^a-z0-9]", "", mid.split("/")[-1].lower())
R1 = slug(f"{MODEL}#random-init-s1#pool-context{CT}")
for L, s in LANGS.items():
    out = f"outputs/activations_xlcost/isect_{s}_{SPLIT}_{R1}"
    if pathlib.Path(f"{out}/meta.json").exists():
        continue
    !python scripts/extract_activations.py run \
      --canonical data/xlcost/{s}_{SPLIT}_isect.jsonl \
      --occurrences outputs/role_occ/isect_{s}_{SPLIT}.jsonl \
      --model-id {MODEL} --label-field role \
      --pool context --context-tokens {CT} --random-init --random-seed 1 \
      --out-dir {out}
for role in ("accumulator","iterator","index_key"):
    for a, b in itertools.permutations(LANGS.values(), 2):
        out = f"outputs/crosslang/probe_{role}_{a}_to_{b}_{R1}.json"
        if pathlib.Path(out).exists():
            continue
        !python scripts/crosslang.py run \
          --train-store outputs/activations_xlcost/isect_{a}_{SPLIT}_{R1} \
          --test-store  outputs/activations_xlcost/isect_{b}_{SPLIT}_{R1} \
          --role {role} --seeds 0 1 --output {out}
!mkdir -p {MK}/crosslang && cp outputs/crosslang/*_{R1}.json {MK}/crosslang/ 2>/dev/null || true
!cp -rn outputs/activations_xlcost/isect_*_{R1} {MK}/stores/ 2>/dev/null || true

In [ ]:
# 7 - Part 2 readout: does the floor move across seeds?
import glob, statistics as st
def floor(sl):
    rs=[]
    for f in glob.glob(f"outputs/crosslang/probe_*_{sl}.json"):
        d=json.loads(open(f).read())
        m=re.match(r"probe_([a-z_]+)_([a-z]+)_to_([a-z]+)_", pathlib.Path(f).name)
        rs.append({"pair":(m.group(2),m.group(3)),"v":d["transfer_macro_f1_mean"]})
    near=[r["v"] for r in rs if "python" not in r["pair"]]
    far=[r["v"] for r in rs if "python" in r["pair"]]
    return (st.mean(near), st.mean(far), len(rs)) if rs else (None,None,0)
S0 = slug(f"{MODEL}#random-init-s0#pool-context{CT}")
for lbl, sl in (("seed 0 (committed)", S0), ("seed 1 (this run)", R1)):
    n,f,k = floor(sl)
    if k: print(f"  {lbl:<20} close {n:.3f}  Python {f:.3f}  effect {f-n:+.3f}  ({k} cells)")
print("\nPaste this cell and cell 5's output back into the chat.")